# Lab 1: Data Visualization, Data Preprocessing, and Statistical Analysis Using Python in Jupyter Notebook

**Name:** Hanuman Sai Chanukya Srinivas Chilamkuri  
**Course:** MSCS 634 - Advanced Big Data and Data Mining  
**Lab Assignment:** Lab 1 - Data Visualization, Preprocessing, and Statistical Analysis

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Create screenshots folder ─────────────────────────────────────────────────
SS = 'screenshots'
os.makedirs(SS, exist_ok=True)

# Helper: save a DataFrame as a PNG table
def save_table(df_or_str, filename, title=''):
    """Render a DataFrame (or plain text) as a tight PNG and save to screenshots/."""
    if isinstance(df_or_str, str):
        lines = df_or_str.strip().split('\n')
        fig_h = max(1.2, len(lines) * 0.28 + 0.6)
        fig, ax = plt.subplots(figsize=(10, fig_h))
        ax.axis('off')
        ax.text(0.01, 0.99, df_or_str, transform=ax.transAxes,
                va='top', ha='left', fontfamily='monospace', fontsize=9)
        if title:
            ax.set_title(title, fontsize=10, fontweight='bold', pad=4)
    else:
        data = df_or_str.reset_index() if not isinstance(df_or_str.index, pd.RangeIndex) else df_or_str
        col_labels = list(data.columns)
        cell_text  = data.astype(str).values.tolist()
        fig_h = max(1.2, len(cell_text) * 0.35 + 0.8)
        fig, ax = plt.subplots(figsize=(max(8, len(col_labels)*1.4), fig_h))
        ax.axis('off')
        tbl = ax.table(cellText=cell_text, colLabels=col_labels,
                       loc='center', cellLoc='center')
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        tbl.scale(1, 1.4)
        if title:
            ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    plt.tight_layout()
    path = os.path.join(SS, filename)
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.close()
    print(f'  Saved → {path}')

# Global plot style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

print('Setup complete. Screenshots will be saved to:', os.path.abspath(SS))

---
## Step 1: Data Collection

In [ ]:
np.random.seed(42)
n = 1000
regions    = ['East', 'West', 'Central', 'South']
categories = ['Technology', 'Furniture', 'Office Supplies']
sub_cats   = {'Technology': ['Phones','Computers','Accessories'],
              'Furniture':  ['Chairs','Tables','Bookcases'],
              'Office Supplies': ['Paper','Binders','Pens']}
ship_modes = ['Standard Class','Second Class','First Class','Same Day']
segments   = ['Consumer','Corporate','Home Office']

order_dates = pd.date_range('2020-01-01','2023-12-31', periods=n)
cat_col     = np.random.choice(categories, n)
sub_col     = [np.random.choice(sub_cats[c]) for c in cat_col]
sales_col   = np.round(np.random.exponential(scale=250, size=n) + 10, 2)
qty_col     = np.random.randint(1, 15, n)
disc_col    = np.round(np.random.choice([0,0.1,0.2,0.3,0.4,0.5], n), 2)
profit_col  = np.round(sales_col * (np.random.uniform(0.05,0.35,n) - disc_col*0.6), 2)

for col in [sales_col, profit_col]:
    idx = np.random.choice(n, int(n*0.05), replace=False); col[idx] = np.nan
outlier_idx = np.random.choice(n, 8, replace=False)
sales_col[outlier_idx] = np.random.uniform(3000, 8000, 8)

df = pd.DataFrame({
    'Order Date':   order_dates,
    'Region':       np.random.choice(regions, n),
    'Segment':      np.random.choice(segments, n),
    'Ship Mode':    np.random.choice(ship_modes, n),
    'Category':     cat_col,
    'Sub-Category': sub_col,
    'Sales':        sales_col,
    'Quantity':     qty_col,
    'Discount':     disc_col,
    'Profit':       profit_col,
})
df.to_csv('superstore_sales.csv', index=False)
print(f'Dataset shape: {df.shape}')

# ── Screenshot SS-01 ──────────────────────────────────────────────────────────
save_table(df.head(), 'SS-01_data_head.png', title='Step 1 – First 5 Rows of Dataset')
df.head()

---
## Step 2: Data Visualization

In [ ]:
# ── 2-A  Scatter: Sales vs Profit ────────────────────────────────────────────
palette = {'Technology':'#4C72B0','Furniture':'#DD8452','Office Supplies':'#55A868'}
fig, ax = plt.subplots(figsize=(10,5))
for cat, grp in df.dropna(subset=['Sales','Profit']).groupby('Category'):
    ax.scatter(grp['Sales'], grp['Profit'], alpha=0.55, s=30, label=cat, color=palette[cat])
ax.set_xlabel('Sales ($)'); ax.set_ylabel('Profit ($)')
ax.set_title('Sales vs. Profit by Category'); ax.legend(title='Category')
plt.tight_layout()
plt.savefig(os.path.join(SS,'SS-02a_scatter.png'), dpi=120)
plt.show()
print('Insight: Technology orders cluster at high-sales/high-profit. Furniture shows many'
      ' high-sales but low/negative-profit orders — indicating heavy discounting.')

In [ ]:
# ── 2-B  Line: Monthly Sales Trend ───────────────────────────────────────────
monthly = (df.dropna(subset=['Sales']).set_index('Order Date')['Sales']
             .resample('ME').sum().reset_index())
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(monthly['Order Date'], monthly['Sales'], color='#4C72B0', linewidth=1.8, marker='o', markersize=3)
ax.fill_between(monthly['Order Date'], monthly['Sales'], alpha=0.15, color='#4C72B0')
ax.set_title('Monthly Sales Trend (2020–2023)'); ax.set_xlabel('Month'); ax.set_ylabel('Total Sales ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(os.path.join(SS,'SS-02b_line.png'), dpi=120)
plt.show()
print('Insight: Consistent Q4 sales spikes every year. Gradual upward baseline 2020–2023.')

In [ ]:
# ── 2-C  Bar: Sales by Region ─────────────────────────────────────────────────
reg_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(reg_sales.index, reg_sales.values,
              color=['#4C72B0','#DD8452','#55A868','#C44E52'], edgecolor='white')
ax.bar_label(bars, fmt='${:,.0f}', padding=4, fontsize=9)
ax.set_title('Total Sales by Region'); ax.set_ylabel('Sales ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(os.path.join(SS,'SS-02c_bar.png'), dpi=120)
plt.show()
print('Insight: West leads in total revenue, followed by East. Central and South trail.')

In [ ]:
# ── 2-D  Histogram: Sales Distribution ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(df['Sales'].dropna(), bins=50, color='#4C72B0', edgecolor='white', linewidth=0.4)
ax.set_title('Distribution of Order Sales'); ax.set_xlabel('Sales ($)'); ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(os.path.join(SS,'SS-02d_histogram.png'), dpi=120)
plt.show()
print('Insight: Heavy right-skew — most orders under $500 with a long tail of high-value outliers.')

In [ ]:
# ── 2-E  Box Plot: Profit by Category ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8,5))
df.dropna(subset=['Profit']).boxplot(column='Profit', by='Category', ax=ax,
    boxprops=dict(color='#4C72B0'), medianprops=dict(color='#DD8452', linewidth=2),
    whiskerprops=dict(color='#4C72B0'), capprops=dict(color='#4C72B0'),
    flierprops=dict(marker='o', markerfacecolor='#C44E52', markersize=4, alpha=0.5))
plt.suptitle('')
ax.set_title('Profit Distribution by Category'); ax.set_xlabel('Category'); ax.set_ylabel('Profit ($)')
plt.tight_layout()
plt.savefig(os.path.join(SS,'SS-02e_boxplot.png'), dpi=120)
plt.show()
print('Insight: Technology has highest median profit. Furniture has most negative-profit outliers.')

In [ ]:
# ── 2-F  Pie: Ship Mode Share ─────────────────────────────────────────────────
ship_counts = df['Ship Mode'].value_counts()
fig, ax = plt.subplots(figsize=(7,5))
wedges, texts, autotexts = ax.pie(ship_counts, labels=ship_counts.index, autopct='%1.1f%%',
    colors=['#4C72B0','#DD8452','#55A868','#C44E52'],
    startangle=140, wedgeprops=dict(edgecolor='white', linewidth=1.5))
for at in autotexts: at.set_fontsize(9)
ax.set_title('Order Share by Ship Mode')
plt.tight_layout()
plt.savefig(os.path.join(SS,'SS-02f_pie.png'), dpi=120)
plt.show()
print('Insight: Standard Class dominates (~40%). Same Day is only ~10% — an upsell opportunity.')

---
## Step 3: Data Preprocessing
### 3-1  Handling Missing Values

In [ ]:
before_txt = '=== Missing Values BEFORE ===\n' + df.isnull().sum().to_string()
print(before_txt)
save_table(before_txt, 'SS-03a_missing_before.png', title='Step 3-1 – Missing Values BEFORE')

In [ ]:
df_clean = df.copy()
df_clean['Sales'].fillna(df_clean['Sales'].mean(), inplace=True)
df_clean['Profit'].fillna(df_clean['Profit'].median(), inplace=True)

after_txt = '=== Missing Values AFTER ===\n' + df_clean.isnull().sum().to_string()
print(after_txt)
save_table(after_txt, 'SS-03b_missing_after.png', title='Step 3-1 – Missing Values AFTER')

### 3-2  Outlier Detection and Removal

In [ ]:
Q1  = df_clean['Sales'].quantile(0.25)
Q3  = df_clean['Sales'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR

iqr_txt = (f'Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}\n'
           f'Lower bound={lower:.2f}  Upper bound={upper:.2f}')
print(iqr_txt)

outliers = df_clean[(df_clean['Sales']<lower)|(df_clean['Sales']>upper)]
print(f'\nOutliers found: {len(outliers)}')
print(outliers[['Sales','Profit','Category']].head(10).to_string())

save_table(iqr_txt + f'\n\nOutliers found: {len(outliers)}\n\n'
           + outliers[['Sales','Profit','Category']].head(10).to_string(),
           'SS-03c_outliers.png', title='Step 3-2 – IQR & Outliers')

df_no_outliers = df_clean[(df_clean['Sales']>=lower)&(df_clean['Sales']<=upper)].copy()
shape_txt = f'Shape before: {df_clean.shape}  →  Shape after: {df_no_outliers.shape}'
print('\n' + shape_txt)
save_table(df_no_outliers[['Sales','Profit']].describe().round(2),
           'SS-03d_after_outlier_removal.png', title='Step 3-2 – Summary After Outlier Removal')

### 3-3  Data Reduction

In [ ]:
before_r = f'Shape: {df_no_outliers.shape}\nColumns: {list(df_no_outliers.columns)}'
print('BEFORE reduction\n' + before_r)
save_table('BEFORE reduction\n' + before_r, 'SS-03e_reduction_before.png', title='Step 3-3 – Before Reduction')

df_sampled = df_no_outliers.sample(frac=0.60, random_state=42).reset_index(drop=True)
df_reduced = df_sampled.drop(columns=['Ship Mode','Sub-Category'])

after_r = f'Shape: {df_reduced.shape}\nColumns: {list(df_reduced.columns)}'
print('\nAFTER reduction\n' + after_r)
save_table('AFTER reduction\n' + after_r, 'SS-03f_reduction_after.png', title='Step 3-3 – After Reduction')

### 3-4  Data Scaling and Discretization

In [ ]:
df_scaled = df_reduced.copy()

save_table(df_scaled[['Sales','Profit']].describe().round(3),
           'SS-03g_scaling_before.png', title='Step 3-4 – Sales & Profit BEFORE Scaling')

mm_scaler  = MinMaxScaler()
std_scaler = StandardScaler()
df_scaled['Sales_MinMax']  = mm_scaler.fit_transform(df_scaled[['Sales']])
df_scaled['Profit_Zscore'] = std_scaler.fit_transform(df_scaled[['Profit']])
df_scaled['Sales_Bin']     = pd.cut(df_scaled['Sales'], bins=4,
                                    labels=['Low','Medium','High','Very High'])

result = df_scaled[['Sales','Sales_MinMax','Profit','Profit_Zscore','Sales_Bin']].head(10)
print(result.to_string())
save_table(result.round(4), 'SS-03h_scaling_after.png',
           title='Step 3-4 – Sales & Profit AFTER Scaling + Discretization')
result

---
## Step 4: Statistical Analysis
### 4-1  General Overview

In [ ]:
import io
buf = io.StringIO(); df_reduced.info(buf=buf)
info_txt = buf.getvalue()
print(info_txt)
save_table(info_txt, 'SS-04a_info.png', title='Step 4-1 – df.info()')

In [ ]:
desc = df_reduced.describe().round(2)
print(desc)
save_table(desc, 'SS-04b_describe.png', title='Step 4-1 – df.describe()')
desc

### 4-2  Central Tendency

In [ ]:
num_cols = ['Sales','Quantity','Discount','Profit']
central = pd.DataFrame({
    'Min':    df_reduced[num_cols].min(),
    'Max':    df_reduced[num_cols].max(),
    'Mean':   df_reduced[num_cols].mean(),
    'Median': df_reduced[num_cols].median(),
    'Mode':   df_reduced[num_cols].mode().iloc[0],
}).round(3)
print(central)
save_table(central, 'SS-04c_central_tendency.png', title='Step 4-2 – Central Tendency Measures')
central

### 4-3  Dispersion Measures

In [ ]:
q1 = df_reduced[num_cols].quantile(0.25)
q3 = df_reduced[num_cols].quantile(0.75)
dispersion = pd.DataFrame({
    'Range':    df_reduced[num_cols].max() - df_reduced[num_cols].min(),
    'Q1':       q1, 'Q3': q3,
    'IQR':      q3 - q1,
    'Variance': df_reduced[num_cols].var(),
    'Std Dev':  df_reduced[num_cols].std(),
}).round(3)
print(dispersion)
save_table(dispersion, 'SS-04d_dispersion.png', title='Step 4-3 – Dispersion Measures')
dispersion

### 4-4  Correlation Analysis

In [ ]:
corr_matrix = df_reduced[num_cols].corr().round(3)
print(corr_matrix)
save_table(corr_matrix, 'SS-04e_corr_matrix.png', title='Step 4-4 – Correlation Matrix')

fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size':11})
ax.set_title('Correlation Matrix – Numerical Features')
plt.tight_layout()
plt.savefig(os.path.join(SS,'SS-04f_corr_heatmap.png'), dpi=120)
plt.show()
print('Insight: Discount negatively correlates with Profit. Sales and Profit are moderately positive.')

In [ ]:
df_scaled.to_csv('superstore_sales_processed.csv', index=False)

saved = sorted(os.listdir(SS))
print(f'\nAll done! {len(saved)} files saved to /{SS}/')
for f in saved:
    print(f'  {f}')